# Vehicle Specification Extraction System
## Optimized for Ford Service Manual Format

This notebook extracts specifications from automotive service manuals in table/symptom chart format.

### What This Does:
- Extracts text from PDF service manuals (Ford, GM, Toyota, etc.)
- Handles symptom charts, tables, and specification lists
- Uses Mistral AI to extract structured specifications
- Exports results to JSON/CSV

## Step 1: Install Required Packages

In [ ]:
!pip install PyMuPDF sentence-transformers faiss-cpu mistralai numpy pandas

## Step 2: Complete Implementation

All code is self-contained - no external files needed.

In [ ]:
import json
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict
import numpy as np
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss
from mistralai import Mistral


@dataclass
class VehicleSpec:
    """Structured representation of a vehicle specification"""
    component: str
    spec_type: str
    value: str
    unit: str
    context: Optional[str] = None
    confidence: Optional[float] = None


class PDFTextExtractor:
    """Handles PDF parsing and text extraction"""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.doc = None

    def extract_text(self) -> List[Dict[str, any]]:
        """Extract text from PDF with page metadata"""
        self.doc = fitz.open(self.pdf_path)
        pages_data = []

        for page_num in range(len(self.doc)):
            page = self.doc[page_num]
            text = page.get_text("text")
            text = self._clean_text(text)

            pages_data.append({
                'page_number': page_num + 1,
                'text': text,
                'char_count': len(text)
            })

        return pages_data

    def _clean_text(self, text: str) -> str:
        """Clean extracted text while preserving structure"""
        # Remove excessive blank lines but keep some structure
        text = re.sub(r'\n\s*\n\s*\n', '\n\n', text)

        # Remove extra spaces within lines
        lines = text.split('\n')
        lines = [re.sub(r' +', ' ', line) for line in lines]
        text = '\n'.join(lines)

        # Remove common footer/header patterns
        text = re.sub(r'Page \d+\s*sur\s*\d+.*?\n', '', text)
        text = re.sub(r'file:///.*?\n', '', text)
        text = re.sub(r'\d{4}-\d{2}-\d{2}\s*$', '', text, flags=re.MULTILINE)

        return text.strip()

    def close(self):
        if self.doc:
            self.doc.close()


class TextChunker:
    """Splits text into semantic chunks optimized for service manuals"""

    def __init__(self, chunk_size: int = 600, overlap: int = 150):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk_by_section(self, pages_data: List[Dict]) -> List[Dict]:
        """Create chunks optimized for symptom charts and tables"""
        chunks = []

        for page_data in pages_data:
            text = page_data['text']
            page_num = page_data['page_number']

            # First try to detect tables/symptom charts
            if self._is_table_like(text):
                table_chunks = self._chunk_table(text, page_num)
                chunks.extend(table_chunks)
            else:
                # Regular section-based chunking
                sections = self._split_by_headers(text)

                for section in sections:
                    if len(section['text']) > self.chunk_size:
                        sub_chunks = self._split_by_paragraphs(section['text'])
                        for sub_chunk in sub_chunks:
                            chunks.append({
                                'text': sub_chunk,
                                'page_number': page_num,
                                'section_title': section.get('title', '')
                            })
                    else:
                        chunks.append({
                            'text': section['text'],
                            'page_number': page_num,
                            'section_title': section.get('title', '')
                        })

        return chunks

    def _is_table_like(self, text: str) -> bool:
        """Detect if text contains table/symptom chart format"""
        # Look for common table indicators
        indicators = [
            'Condition',
            'Possible Sources',
            'Action',
            'Symptom Chart',
            'Specification',
            'TIGHTEN',
            'INSTALL',
            'REFER to Section'
        ]
        return any(indicator in text for indicator in indicators)

    def _chunk_table(self, text: str, page_num: int) -> List[Dict]:
        """Chunk table-like content keeping rows together"""
        chunks = []

        # Split by major sections within the table
        # Look for patterns like "Condition" followed by content
        lines = text.split('\n')
        current_chunk = ''
        section_title = ''

        for i, line in enumerate(lines):
            # Detect section headers in symptom charts
            if any(header in line for header in ['Symptom Chart', 'Inspection and Verification']):
                section_title = line.strip()
                if current_chunk:
                    chunks.append({
                        'text': current_chunk.strip(),
                        'page_number': page_num,
                        'section_title': section_title
                    })
                current_chunk = line + '\n'
            # Keep table rows together
            elif len(current_chunk) + len(line) < self.chunk_size:
                current_chunk += line + '\n'
            else:
                if current_chunk:
                    chunks.append({
                        'text': current_chunk.strip(),
                        'page_number': page_num,
                        'section_title': section_title
                    })
                current_chunk = line + '\n'

        # Add final chunk
        if current_chunk:
            chunks.append({
                'text': current_chunk.strip(),
                'page_number': page_num,
                'section_title': section_title
            })

        return chunks if chunks else [{'text': text, 'page_number': page_num, 'section_title': ''}]

    def _split_by_headers(self, text: str) -> List[Dict]:
        """Split text by section headers"""
        # Patterns for headers: all caps, numbered sections
        header_patterns = [
            r'^([A-Z][A-Z\s]+)$',  # All caps
            r'^(\d+\.\s+[A-Z].*?)$',  # Numbered sections
        ]

        sections = []
        current_section = {'title': '', 'text': ''}

        for line in text.split('\n'):
            is_header = False
            for pattern in header_patterns:
                if re.match(pattern, line.strip()) and len(line.strip()) < 100:
                    is_header = True
                    break

            if is_header:
                if current_section['text']:
                    sections.append(current_section)
                current_section = {'title': line.strip(), 'text': ''}
            else:
                current_section['text'] += line + '\n'

        if current_section['text']:
            sections.append(current_section)

        return sections if sections else [{'title': '', 'text': text}]

    def _split_by_paragraphs(self, text: str) -> List[str]:
        """Split text by paragraphs with overlap"""
        paragraphs = text.split('\n\n')
        chunks = []
        current_chunk = ''

        for para in paragraphs:
            if len(current_chunk) + len(para) < self.chunk_size:
                current_chunk += para + '\n\n'
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para + '\n\n'

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks if chunks else [text]


class EmbeddingManager:
    """Manages embeddings and vector store"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.chunks = None
        self.embeddings = None

    def create_embeddings(self, chunks: List[Dict]) -> np.ndarray:
        """Create embeddings for all chunks"""
        texts = [chunk['text'] for chunk in chunks]
        self.chunks = chunks

        print(f"Creating embeddings for {len(texts)} chunks...")
        self.embeddings = self.model.encode(texts, show_progress_bar=True)

        return self.embeddings

    def build_index(self, embeddings: np.ndarray):
        """Build FAISS index for fast retrieval"""
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype('float32'))
        print(f"Built FAISS index with {self.index.ntotal} vectors")

    def search(self, query: str, k: int = 5) -> List[Tuple[Dict, float]]:
        """Search for relevant chunks"""
        query_embedding = self.model.encode([query])
        distances, indices = self.index.search(query_embedding.astype('float32'), k)

        results = []
        for idx, distance in zip(indices[0], distances[0]):
            results.append((self.chunks[idx], float(distance)))

        return results


class SpecificationExtractor:
    """Uses LLM to extract structured specifications from text"""

    def __init__(self, api_key: str, model: str = "mistral-large-latest"):
        self.client = Mistral(api_key=api_key)
        self.model = model

    def extract_specs(self, query: str, contexts: List[Dict]) -> List[VehicleSpec]:
        """Extract specifications using LLM with retrieved context"""
        context_text = self._prepare_context(contexts)
        prompt = self._create_extraction_prompt(query, context_text)

        response = self.client.chat.complete(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are an expert at extracting vehicle specifications from service manuals, including torque values, fluid capacities, tire pressures, and other technical specifications. Always return valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1
        )

        specs = self._parse_llm_response(response.choices[0].message.content)
        return specs

    def _prepare_context(self, contexts: List[Dict]) -> str:
        """Prepare context from retrieved chunks"""
        context_parts = []
        for i, ctx in enumerate(contexts, 1):
            section = ctx.get('section_title', '')
            section_info = f" - {section}" if section else ""
            context_parts.append(
                f"[Context {i} - Page {ctx['page_number']}{section_info}]\n{ctx['text']}\n"
            )
        return "\n".join(context_parts)

    def _create_extraction_prompt(self, query: str, context: str) -> str:
        """Create prompt for specification extraction from service manuals"""
        return f"""Given the following query and context from a vehicle service manual, extract all relevant specifications.

The context may contain:
- Symptom charts with conditions and actions
- Torque specifications (in Nm, lb-ft, etc.)
- Fluid capacities (in L, qt, etc.)
- Tire pressures (in psi, kPa, etc.)
- Clearances and gaps (in mm, inches, etc.)
- Procedures with specific values

Query: {query}

Context:
{context}

Extract specifications in the following JSON format:
[
  {{
    "component": "Name of the component (e.g., 'Wheel Nut', 'Engine Oil', 'Front Tire')",
    "spec_type": "Type of specification (e.g., 'Torque', 'Capacity', 'Pressure', 'Clearance', 'Gap')",
    "value": "Numeric value only (e.g., '110', '4.5', '35')",
    "unit": "Unit of measurement (e.g., 'Nm', 'lb-ft', 'L', 'qt', 'psi', 'mm', 'in')",
    "context": "Brief additional context (e.g., 'Front wheel', 'With filter change', 'Cold tire pressure')"
  }}
]

Rules:
1. Only extract specifications that are explicitly stated with numeric values
2. Separate numeric values from units (e.g., "110 Nm" → value: "110", unit: "Nm")
3. Use standard unit abbreviations (Nm, lb-ft, L, qt, psi, kPa, mm, in)
4. Extract ALL specifications mentioned, even if in procedures like "TIGHTEN to X Nm"
5. For torque specs in tables, extract each fastener type separately
6. If multiple specifications match the query, include all of them
7. Return an empty list [] if no relevant specifications are found
8. Ensure the response is valid JSON with no additional text

Response:"""

    def _parse_llm_response(self, response: str) -> List[VehicleSpec]:
        """Parse LLM response into VehicleSpec objects"""
        try:
            # Try to extract JSON array from response
            json_match = re.search(r'\[.*\]', response, re.DOTALL)
            if json_match:
                json_str = json_match.group()
                specs_data = json.loads(json_str)

                specs = []
                for spec_dict in specs_data:
                    specs.append(VehicleSpec(
                        component=spec_dict.get('component', ''),
                        spec_type=spec_dict.get('spec_type', ''),
                        value=spec_dict.get('value', ''),
                        unit=spec_dict.get('unit', ''),
                        context=spec_dict.get('context'),
                        confidence=spec_dict.get('confidence')
                    ))
                return specs
            else:
                return []
        except json.JSONDecodeError as e:
            print(f"Error parsing LLM response: {e}")
            print(f"Response: {response[:200]}...")
            return []


class VehicleSpecExtractionPipeline:
    """Main pipeline orchestrating the extraction process"""

    def __init__(self, pdf_path: str, api_key: str):
        self.pdf_path = pdf_path
        self.api_key = api_key

        self.pdf_extractor = PDFTextExtractor(pdf_path)
        self.chunker = TextChunker(chunk_size=600, overlap=150)  # Larger chunks for tables
        self.embedding_manager = EmbeddingManager()
        self.spec_extractor = SpecificationExtractor(api_key)

        self.is_indexed = False

    def index_document(self):
        """Index the PDF document"""
        print("Step 1: Extracting text from PDF...")
        pages_data = self.pdf_extractor.extract_text()
        print(f"Extracted {len(pages_data)} pages")

        print("\nStep 2: Chunking text (optimized for service manuals)...")
        chunks = self.chunker.chunk_by_section(pages_data)
        print(f"Created {len(chunks)} chunks")

        print("\nStep 3: Creating embeddings...")
        embeddings = self.embedding_manager.create_embeddings(chunks)

        print("\nStep 4: Building vector index...")
        self.embedding_manager.build_index(embeddings)

        self.is_indexed = True
        print("\n✓ Document indexed successfully!")

    def query(self, query: str, top_k: int = 5) -> List[VehicleSpec]:
        """Query the system for specifications"""
        if not self.is_indexed:
            raise RuntimeError("Document not indexed. Call index_document() first.")

        print(f"\nQuerying: {query}")
        print(f"Retrieving top {top_k} relevant chunks...")
        results = self.embedding_manager.search(query, k=top_k)

        contexts = [result[0] for result in results]

        print("Extracting specifications with LLM...")
        specs = self.spec_extractor.extract_specs(query, contexts)

        return specs

    def export_to_json(self, specs: List[VehicleSpec], output_path: str):
        """Export specifications to JSON"""
        specs_dict = [asdict(spec) for spec in specs]
        with open(output_path, 'w') as f:
            json.dump(specs_dict, f, indent=2)
        print(f"Results exported to {output_path}")

    def export_to_csv(self, specs: List[VehicleSpec], output_path: str):
        """Export specifications to CSV"""
        import csv

        if not specs:
            print("No specifications to export")
            return

        with open(output_path, 'w', newline='') as f:
            fieldnames = ['component', 'spec_type', 'value', 'unit', 'context', 'confidence']
            writer = csv.DictWriter(f, fieldnames=fieldnames)

            writer.writeheader()
            for spec in specs:
                writer.writerow(asdict(spec))

        print(f"Results exported to {output_path}")

    def close(self):
        """Clean up resources"""
        self.pdf_extractor.close()


print("✓ All classes loaded successfully!")
print("\nOptimizations for Ford/GM service manuals:")
print("  - Table detection for symptom charts")
print("  - Larger chunks (600 chars) to keep procedures together")
print("  - Enhanced torque specification extraction")
print("  - Better handling of TIGHTEN/INSTALL instructions")

✓ All classes loaded successfully!

Optimizations for Ford/GM service manuals:
  - Table detection for symptom charts
  - Larger chunks (600 chars) to keep procedures together
  - Enhanced torque specification extraction
  - Better handling of TIGHTEN/INSTALL instructions


## Step 3: Configuration

Set your Mistral API key and PDF path.

In [ ]:
import os

MISTRAL_API_KEY = "OYzdmUUPY5keaI3Am8kQWitMICTDNyxM"

# Path to  service manual PDF
PDF_PATH = "sample-service-manual.pdf"

# Verify files exist
if not os.path.exists(PDF_PATH):
    print(f" Warning: PDF file not found at {PDF_PATH}")
    print("Please update PDF_PATH to point to your service manual.")
else:
    print(f"PDF file found: {PDF_PATH}")

PDF file found: sample-service-manual.pdf


## Step 4: Initialize and Index Document

In [ ]:
# Initialize pipeline
pipeline = VehicleSpecExtractionPipeline(PDF_PATH, MISTRAL_API_KEY)

# Index the document
pipeline.index_document()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 1: Extracting text from PDF...
Extracted 852 pages

Step 2: Chunking text (optimized for service manuals)...
Created 1535 chunks

Step 3: Creating embeddings...
Creating embeddings for 1535 chunks...


Batches:   0%|          | 0/48 [00:00<?, ?it/s]


Step 4: Building vector index...
Built FAISS index with 1535 vectors

✓ Document indexed successfully!


## Step 5: Example Queries for Ford Service Manual

These queries are optimized for the symptom chart format.

In [ ]:
# Example query - Wheel nut torque
query = "wheel nut torque specification"
specs = pipeline.query(query, top_k=5)

# Display results
print(f"\n{'='*60}")
print(f"Found {len(specs)} specification(s) for: {query}")
print('='*60)

for i, spec in enumerate(specs, 1):
    print(f"\n{i}. Component: {spec.component}")
    print(f"   Type: {spec.spec_type}")
    print(f"   Value: {spec.value} {spec.unit}")
    if spec.context:
        print(f"   Context: {spec.context}")


Querying: wheel nut torque specification
Retrieving top 5 relevant chunks...
Extracting specifications with LLM...

Found 2 specification(s) for: wheel nut torque specification

1. Component: Wheel Nut
   Type: Torque
   Value: 150 lb-ft
   Context: 6 or 7 required per wheel

2. Component: Wheel Nut
   Type: Torque
   Value: 203 Nm
   Context: Converted from 150 lb-ft (6 or 7 required per wheel)


## Step 6: Try Multiple Queries

In [ ]:
# Queries optimized for service manual format
queries = [
    "wheel nut torque",
    "tire pressure specifications",
    "stabilizer bracket to frame bolts torque",
    "suspension fastener torque",
    "shock absorber specifications",
    "alignment specifications"
]

all_specs = []

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)

    specs = pipeline.query(query, top_k=5)
    all_specs.extend(specs)

    if specs:
        for spec in specs:
            print(f"   {spec.component}: {spec.value} {spec.unit} ({spec.spec_type})")
            if spec.context:
                print(f"    Note: {spec.context}")
    else:
        print(" No specifications found")

print(f"\n{'='*60}")
print(f"Total specifications extracted: {len(all_specs)}")
print('='*60)


Query: wheel nut torque

Querying: wheel nut torque
Retrieving top 5 relevant chunks...
Extracting specifications with LLM...
Error parsing LLM response: Extra data: line 17 column 1 (char 397)
Response: ```json
[
  {
    "component": "Wheel Nut",
    "spec_type": "Torque",
    "value": "201",
    "unit": "Nm",
    "context": "Wheel nut torque specification (derived from lb-ft value)"
  },
  {
    "co...
 No specifications found

Query: tire pressure specifications

Querying: tire pressure specifications
Retrieving top 5 relevant chunks...
Extracting specifications with LLM...
   Tire: Refer to VC label psi (Pressure)
    Note: Recommended inflation pressure (cold tire pressure) as listed on the Vehicle Certification (VC) label located on the driver door jamb

Query: stabilizer bracket to frame bolts torque

Querying: stabilizer bracket to frame bolts torque
Retrieving top 5 relevant chunks...
Extracting specifications with LLM...
   Stabilizer bar bracket to frame bolts: 63 Nm (Torqu

## Step 7: View Results as DataFrame

In [ ]:
import pandas as pd

if all_specs:
    df = pd.DataFrame([asdict(spec) for spec in all_specs])
    df = df[['component', 'spec_type', 'value', 'unit', 'context']]

    print("\nExtracted Specifications:")
    display(df)

    print(f"\nSummary:")
    print(f"Total specifications: {len(df)}")
    print(f"Unique components: {df['component'].nunique()}")
    print(f"\nSpecification types:")
    print(df['spec_type'].value_counts())
else:
    print("No specifications to display")


Extracted Specifications:


,component,spec_type,value,unit,context
0,Tire,Pressure,Refer to VC label,psi,Recommended inflation pressure (cold tire pres...
1,Stabilizer bar bracket to frame bolts,Torque,63,Nm,Installation torque for stabilizer bar bracket...
2,Stabilizer bar bracket to frame bolts,Torque,46,lb-ft,Installation torque for stabilizer bar bracket...
3,Shock absorber lower bolt,Torque,110,Nm,"All vehicles, shock absorber installation"
4,Shock absorber lower nut,Torque,110,Nm,"All vehicles, shock absorber installation"
5,Shock absorber upper bolt,Torque,110,Nm,"All vehicles, shock absorber installation"
6,Shock absorber upper nut,Torque,110,Nm,"All vehicles, shock absorber installation"
7,Shock absorber shield bolt,Torque,10,Nm,"All vehicles, 3 required for shock absorber sh..."
8,Front Toe,Alignment,,,Refer to Alignment Specifications in the Speci...
9,Caster and Camber,Alignment,,,Refer to Alignment Specifications in the Speci...



Summary:
Total specifications: 15
Unique components: 11

Specification types:
spec_type
Torque       9
Clearance    3
Alignment    2
Pressure     1
Name: count, dtype: int64


## Step 8: Export Results

In [ ]:
if all_specs:
    pipeline.export_to_json(all_specs, "extracted_specs.json")
    pipeline.export_to_csv(all_specs, "extracted_specs.csv")

    print("\n Results exported successfully!")
    print("  - JSON: extracted_specs.json")
    print("  - CSV: extracted_specs.csv")
else:
    print("No specifications to export")

Results exported to extracted_specs.json
Results exported to extracted_specs.csv

 Results exported successfully!
  - JSON: extracted_specs.json
  - CSV: extracted_specs.csv


## Step 9: Custom Query

In [ ]:
# Try your own query
custom_query = "TIGHTEN specifications"

specs = pipeline.query(custom_query, top_k=7)
print(f"\nResults for: {custom_query}")
print('='*60)

if specs:
    for spec in specs:
        print(f"\n {spec.component}")
        print(f"  {spec.spec_type}: {spec.value} {spec.unit}")
        if spec.context:
            print(f"  {spec.context}")
else:
    print("No specifications found. Try:")
    print("  - 'torque specifications'")
    print("  - 'TIGHTEN bolts'")
    print("  - 'wheel specifications'")
    print("  - 'suspension components'")


Querying: TIGHTEN specifications
Retrieving top 7 relevant chunks...
Extracting specifications with LLM...

Results for: TIGHTEN specifications

 Unspecified fastener
  Torque: 175 Nm
  Step 2, Page 54

 Unspecified fastener
  Torque: 129 lb-ft
  Equivalent of 175 Nm, Step 2, Page 54

 Unspecified fastener
  Torque: 115 Nm
  Step 3, Page 54

 Unspecified fastener
  Torque: 85 lb-ft
  Equivalent of 115 Nm, Step 3, Page 54

 Pinion shaft bolt
  Torque: 30 Nm
  Step 5, Page 492; coat threads with sealer if new bolt unavailable

 Pinion shaft bolt
  Torque: 22 lb-ft
  Equivalent of 30 Nm, Step 5, Page 492

 Ring gear and carrier bolts
  Torque: 128 Nm
  Step 7, Page 492

 Ring gear and carrier bolts
  Torque: 95 lb-ft
  Equivalent of 128 Nm, Step 7, Page 492

 Unspecified fastener
  Torque: 63 Nm
  Step 5, Page 40

 Unspecified fastener
  Torque: 46 lb-ft
  Equivalent of 63 Nm, Step 5, Page 40

 Unspecified fastener
  Torque: 115 Nm
  Step 6, Page 40

 Unspecified fastener
  Torque: 85 lb

## Step 10: Inspect Retrieved Contexts

In [ ]:
# See what text was retrieved
inspect_query = "wheel nut torque"

results = pipeline.embedding_manager.search(inspect_query, k=3)

print(f"Query: {inspect_query}\n")
print("Retrieved Contexts:\n")

for i, (context, distance) in enumerate(results, 1):
    print(f"[Context {i}] (Distance: {distance:.4f})")
    print(f"Page: {context['page_number']}")
    print(f"Section: {context.get('section_title', 'N/A')}")
    print(f"Text:\n{context['text'][:400]}...")
    print("-" * 80)

Query: wheel nut torque

Retrieved Contexts:

[Context 1] (Distance: 0.7421)
Page: 196
Section: 
Text:
Wheel nut (6 or 7 required) 
3 
—
Wheel and tire assembly 
2014 F-150 Workshop Manual...
--------------------------------------------------------------------------------
[Context 2] (Distance: 0.8429)
Page: 24
Section: 
Text:
Torque Specifications 
SECTION 204-01A: Front Suspension — Rear Wheel Drive (RWD) 
2014 F-150 Workshop Manual 
SPECIFICATIONS 
Procedure revision date: 10/25/2013 
Description 
Nm lb-ft lb-in 
Brake disc shield bolts 
17 
—
150 
Brake hose bracket bolt 
12 
—
106 
Lower arm forward and rearward nuts 
350 258 
—
Lower ball joint nut 
175 129 
—
Shock absorber lower nuts 
90 
66 
—
Shock absorber up...
--------------------------------------------------------------------------------
[Context 3] (Distance: 0.9228)
Page: 128
Section: 
Text:
Torque Specifications 
a Refer to the procedure in this section 
SECTION 204-02: Rear Suspension 
2014 F-150 Workshop Manual 
SPE

## Step 11: Cleanup

In [ ]:
pipeline.close()
print(" Pipeline closed successfully")

 Pipeline closed successfully
